In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# Remove users with less than 200 ratings and books with less than 100 ratings
rating_counts = df_ratings['user'].value_counts()
filtered_users = rating_counts[rating_counts >= 200].index

isbn_counts = df_ratings['isbn'].value_counts()
filtered_books = isbn_counts[isbn_counts >= 100].index

df_ratings = df_ratings[(df_ratings['user'].isin(filtered_users)) & (df_ratings['isbn'].isin(filtered_books))]

In [ ]:
# Merge the two dataframes, clean the data, and pivot the df
df = pd.merge(df_ratings, df_books, on='isbn')
df = df.drop_duplicates(['user', 'title'])
df = df.pivot(columns='user', index='title', values='rating').fillna(0)
df.info()

In [ ]:
# Nearest Neighbors
neigh = NearestNeighbors(metric='cosine', algorithm='brute')
neigh.fit(df)

In [ ]:
def get_recommends(book = ""):
    """
    Returns a list of 5 similar books with their distances from the book argument.
    """
    # Get the row of the book and flatten it out
    x = df.loc[book].array.reshape(1, -1)
    # kneighbors returns two arrays containing the lengths to the points and
    # their indices
    neigh_dist, neigh_ind = neigh.kneighbors(x, n_neighbors=6)
    five_books = []
    # Loop over each of the distances and indices
    for dist, ind in zip(neigh_dist[0], neigh_ind[0]):
        # Append the name of the book and the distance
        rec_book = df.index[ind]
        if rec_book != book:
            five_books.append([rec_book, dist])
    recommended_books = [book, five_books[::-1]]
    return recommended_books

In [ ]:
# Provided test
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False

test_book_recommendation()